In [7]:

! python --version

Python 3.14.6


In [15]:
! gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=1tOOSGUDBDUAGsYlb213HCPTPI6EdF&access_type=offline&code_challenge=vFBzESTfXSq3VguwbJ-5Hq8LTqCiO8uQqNnJ1qW3VZs&code_challenge_method=S256


Credentials saved to file: [C:\Users\yogen\AppData\Roaming\gcloud\application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "gen-lang-client-0301336617" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


### Deploy Few Models using Model Garden

In [9]:
# Use the environment variable if the user doesn't provide Project ID.
import os
import vertexai
from dotenv import load_dotenv
load_dotenv()
from vertexai import model_garden
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT_ID")
LOCATION = os.environ.get("GOOGLE_CLOUD_REGION", "us-central1")
vertexai.init(project=PROJECT_ID, location=LOCATION)


In [10]:

def print_models(data_list: list[str]) -> None:
    print(" --- Models available --- ")
    print("\n")
    print(f"🔢 Total models: {len(data_list)} 🔢\n")  # Print the count here

    for i, item in enumerate(data_list):
        print(f"  {item} \n")

In [11]:

model_garden_models = model_garden.list_deployable_models(model_filter="gemma", list_hf_models=True)

print_models(model_garden_models)


 --- Models available --- 


🔢 Total models: 167 🔢

  google/gemma-2b 

  google/gemma-2b-it 

  google/gemma-7b 

  google/gemma-7b-it 

  google/gemma-1.1-2b-it 

  google/gemma-1.1-7b-it 

  google/gemma-2-9b 

  google/gemma-2-9b-it 

  google/gemma-2-27b 

  google/gemma-2-27b-it 

  google/gemma-2-2b 

  google/gemma-2-2b-it 

  modelspace/gemmax2-28-2b-v0.1 

  baai/bge-reranker-v2-gemma 

  metin/gemma-2-2b-tr-knowledge-graph 

  unsloth/gemma-3-4b-it-unsloth-bnb-4bit 

  mlabonne/gemma-3-12b-it-abliterated 

  mlabonne/gemma-3-27b-it-abliterated 

  google/gemma-3-4b-it 

  google/gemma-3-12b-it 

  google/gemma-3-1b-it 

  google/gemma-3-27b-it 

  google/gemma-3-27b-pt 

  google/gemma-3-12b-pt 

  google/gemma-3-4b-pt 

  google/gemma-3-1b-pt 

  gaunernst/gemma-3-27b-it-int4-awq 

  allura-org/gemma-3-glitter-12b 

  toastypigeon/gemma-3-starshine-12b 

  allura-org/gemma-3-glitter-4b 

  neo4j/text-to-cypher-gemma-3-4b-instruct-2025.04.0 

  neo4j/text-to-cypher-gemma-3-2

In [12]:
model_id = "google/gemma3@gemma-3-1b-it"
gemma_model = model_garden.OpenModel(model_id)

deploy_options = gemma_model.list_deploy_options(concise=True)
print(deploy_options)



[Option 1: vLLM 32K context]
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/vertex-vision-model-garden-dockers/pytorch-vllm-serve:20260220_0916_RC01",
    machine_type="g4-standard-48",
    accelerator_type="NVIDIA_RTX_PRO_6000",
    accelerator_count=1,

[Option 2: vLLM 32K context]
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/vertex-vision-model-garden-dockers/pytorch-vllm-serve:20250430_0916_RC00_maas",
    machine_type="a2-ultragpu-1g",
    accelerator_type="NVIDIA_A100_80GB",
    accelerator_count=1,

[Option 3: vLLM 32K context]
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/vertex-vision-model-garden-dockers/pytorch-vllm-serve:20250430_0916_RC00_maas",
    machine_type="a3-highgpu-1g",
    accelerator_type="NVIDIA_H100_80GB",
    accelerator_count=1,

[Option 4: vLLM 32K context]
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/vertex-vision-model-garden-dockers/pytorch-vllm-serve:20250430_0916_RC00_maas",
    machine_type=

In [13]:
gemma_endpoint = gemma_model.deploy(
    machine_type="g2-standard-12",
    accelerator_type="NVIDIA_L4",
    accelerator_count=1,
    min_replica_count=1,
    max_replica_count=1,
    endpoint_display_name="gemma_model_endpoint",
    model_display_name="gemma_model_self",
    deploy_request_timeout=3 * 60 * 60,
)

Deploying model: google/gemma3@gemma-3-1b-it
LRO: projects/778374384781/locations/us-central1/operations/6536845670806978560
Start time: 2026-07-05 19:13:33.638385
End time: 2026-07-05 19:15:18.464192
Endpoint: projects/778374384781/locations/us-central1/endpoints/mg-endpoint-42680664-f30f-4f9e-8784-e9b28f8faeab


In [16]:
prediction = gemma_endpoint.predict(
    instances=[{"prompt": "Tell me about Google Cloud ", "temperature": 0.0001, "max_tokens": 150}]
)
print(prediction.predictions[0])

Prompt:
Tell me about Google Cloud
Output:
3.0.

Okay, let's dive into Google Cloud 3.0. It's a significant evolution of Google Cloud Platform (GCP), built around the concept of "Cloud Native."  Here's a breakdown of what it is, its key features, and why it's important:

**1. What is Google Cloud 3.0?**

* **Shift from Traditional to Cloud Native:**  Google Cloud 3.0 isn't just a new version of GCP. It's a fundamental shift in how you think about and build applications on GCP. It's designed to support the principles of Cloud Native –  agility, scalability, resilience, and observability – all crucial for modern, rapidly


In [ ]:

delete_endpoints = True

if delete_endpoints:
    gemma_endpoint.delete(force=True)

     